# Dispersive readout: from the Lindblad equation to the IQ clouds

How a voltage signal on a transmission line turns into a measurement of the qubit state,
and where the emission matrix $E$ of the hidden Markov model (`HMM_redout.ipynb`) comes
from.

The chain of reasoning is:

1. A drive on the readout resonator, plus photon loss, is a **Lindblad** problem.
2. Because everything is linear, the equation for $\langle a\rangle$ **closes** — one
   complex number describes the whole cavity.
3. The qubit shifts the resonance by $\pm\chi$, so that number takes two different values,
   $\alpha_g$ and $\alpha_e$.
4. Vacuum plus amplifier noise smears each into a **cloud** in the IQ plane.
5. How well the two clouds separate *is* the readout fidelity, i.e. $E$.

Reference for the experiment being modelled: Dixit et al., **PRL 126, 141302 (2021)**,
where the readout is integrated for $3\,\mu s$ inside a $t_m = 10\,\mu s$ cycle, with a
HEMT amplifier (the JPA in the fridge was not used) setting the fidelity to
$F_{g\mathcal{G}} = 95.8\%$, $F_{e\mathcal{E}} = 95.3\%$.

In [ ]:
import qutip as qt
import matplotlib.pyplot as plt
import numpy as np

import tqdm

## 1. The model

In the frame rotating at the drive frequency $\omega_d$, a readout resonator dispersively
coupled to a qubit and driven through the input line is

$$H = \Delta_r\, a^\dagger a + \chi\, a^\dagger a\, \sigma_z + \varepsilon\,(a + a^\dagger),
\qquad \Delta_r = \omega_r - \omega_d$$

with photon loss described by the Lindblad dissipator $\mathcal{D}[\sqrt{\kappa}\,a]$.

* $\varepsilon$ — the drive. This is the voltage signal on the transmission line: it
  couples to the charge on the resonator in a 2D geometry, or to the mode's electric field
  at the antenna in a 3D cavity. Either way the coupling operator is $a + a^\dagger$, so
  only the *value* of $\varepsilon$ depends on the geometry, never the form.
* $\kappa$ — how **wide** the resonance is (how fast photons leak out).
* $\chi$ — how far the resonance **moves** when the qubit flips.

The qubit is not dynamical here: $\sigma_z$ commutes with everything in $H$, so we can fix
$\sigma_z \to s = \pm 1$ and solve the cavity problem twice. Define

$$\Delta_\text{eff} = -\delta_r + s\,\chi, \qquad s = -1 \;\text{for}\; |g\rangle, \qquad s = +1 \;\text{for}\; |e\rangle$$

In [ ]:
# Units: frequency as 2*pi * MHz, time in microseconds.
# These are generic readout-resonator numbers, NOT the storage cavity: the storage
# cavity of the paper has 2*chi = 2*pi*1.13 MHz and appears in HMM_redout.ipynb as
# the discrete hidden variable n, never as an operator.

chi     = 2.0 * np.pi * 2.0   # dispersive shift: how far the resonance MOVES
kappa   = 2.0 * np.pi * 5.0   # resonator linewidth: how WIDE the resonance is
epsilon = 2.0 * np.pi * 4.0   # drive amplitude (the voltage on the input line)
delta_r = 0.0                 # probe detuning; 0 = midway between the two resonances

print(f"chi/kappa = {chi/kappa:.2f}   (the optimum is 0.5, see section 5)")

## 2. From Lindblad to a single complex number

The equation of motion of any observable under a Lindblad master equation is

$$\frac{d\langle O\rangle}{dt} = i\big\langle[H,O]\big\rangle + \big\langle \mathcal{D}^\dagger[O]\big\rangle,
\qquad
\mathcal{D}^\dagger[O] = \kappa\Big(a^\dagger O a - \tfrac{1}{2}\{a^\dagger a,\, O\}\Big)$$

Take $O = a$.

**The commutator.** With $H = \Delta_\text{eff}\,a^\dagger a + \varepsilon(a + a^\dagger)$ and
the two elementary results $[a^\dagger a, a] = -a$ and $[a + a^\dagger, a] = [a^\dagger,a] = -1$,

$$[H, a] = -\Delta_\text{eff}\,a - \varepsilon
\qquad\Longrightarrow\qquad
i\langle[H,a]\rangle = -i\Delta_\text{eff}\langle a\rangle - i\varepsilon$$

**The dissipator.**

$$\mathcal{D}^\dagger[a] = \kappa\Big(a^\dagger a a - \tfrac12 a^\dagger a a - \tfrac12 a a^\dagger a\Big)
= \frac{\kappa}{2}\big(a^\dagger a - a a^\dagger\big)a
= \frac{\kappa}{2}[a^\dagger, a]\,a
= -\frac{\kappa}{2}\,a$$

**Putting them together:**

$$\boxed{\;\frac{d\langle a\rangle}{dt} = -i\,\Delta_\text{eff}\,\langle a\rangle - i\varepsilon - \frac{\kappa}{2}\langle a\rangle\;}$$

### Why this is the whole story

The right-hand side contains **only** $\langle a\rangle$ — no $\langle a^\dagger a a\rangle$,
no higher moments. The equation *closes*. That happens because $H$ is at most quadratic and
the dissipator is linear in $a$. Add a Kerr term $K a^\dagger a^\dagger a a$ and
$\langle a^\dagger a a\rangle$ appears on the right, the hierarchy never terminates, and no
single number can describe the cavity.

Note also that this ODE is **exact for any state**. It is just the classical equation for a
driven, damped oscillator: $\langle a\rangle$ is, up to constants, the mean electric field.
Nothing quantum has happened yet.

## 3. Why the state is coherent — and why that matters

The ODE above gives one number. Turning that number into *the state* needs a separate
argument, because **$\langle a\rangle$ does not determine a state**: the vacuum, a Fock
state $|3\rangle$ and a thermal state all have $\langle a\rangle = 0$ and are nothing alike.

Two arguments, either is enough:

**The quantum jump does nothing.** A coherent state is *defined* as the eigenstate of the
annihilation operator, $a|\alpha\rangle = \alpha|\alpha\rangle$. So the jump term of the
dissipator, which acts as $a$, leaves $|\alpha\rangle$ untouched: **losing a photon does not
change a coherent state**. Meanwhile the linear drive generates displacements
($e^{-i\varepsilon(a+a^\dagger)t} = D(\cdot)$, and $D(\beta)D(\alpha) = D(\alpha+\beta)$ up
to a phase) and the detuning generates rotations, $e^{-i\Delta a^\dagger a t}|\alpha\rangle
= |\alpha e^{-i\Delta t}\rangle$. None of the three operations leaves the coherent manifold.

**The Gaussian argument.** A quadratic $H$ with linear loss preserves Gaussian states, so
the state is fixed by its first and second moments. There are no $a^2$ or $a^{\dagger 2}$
terms — no squeezing — so the second moments stay at their vacuum values. A Gaussian state
with vacuum second moments and first moment $\alpha$ *is* $|\alpha\rangle$.

Only now is it legitimate to say $\alpha = \langle a\rangle$: the identity
$\langle\alpha|a|\alpha\rangle = \alpha$ is a definition of the *label* of a coherent state,
and it describes the full state only because we established the state is coherent.

**The physical reading:** a coherent state is *classical field + vacuum noise*. The centre of
each IQ cloud is $\alpha$, the classical part; the width is the vacuum fluctuation
(variance $1/4$ per quadrature) plus whatever the amplifier adds. Section 6 builds exactly
that, centre and width separately.

In [ ]:
# The claims of sections 2 and 3, checked against the full master equation.
N_photons = 30
a = qt.destroy(N_photons)
adag = a.dag()

def H_ef(args):
    """Effective dispersive Hamiltonian for ONE fixed qubit state.

    sigma_z commutes with everything in H, so instead of carrying the qubit in a
    tensor product we solve the cavity problem twice, once per eigenvalue
    s = args['sign'] = -1 (|g>) or +1 (|e>).

    In the frame rotating at the drive, the cavity sees
        Delta_eff = -delta_r + s*chi
    and the drive enters linearly as epsilon (a + a_dag).
    """
    delta_eff = -args['delta_r'] + args['sign'] * args['chi']
    return delta_eff * adag * a + args['epsilon'] * (a + adag)

# The closed form we are about to derive in section 4, written out here so this
# check stands on its own:
alpha_ss = lambda s: -1j * epsilon / (kappa/2.0 + 1j * (-delta_r + s*chi))

ts = np.linspace(0, 2.0, 201)
for s, name in [(-1, "|g>"), (+1, "|e>")]:
    predicted = alpha_ss(s)
    H = H_ef({'delta_r': delta_r, 'chi': chi, 'epsilon': epsilon, 'sign': s})
    res = qt.mesolve(H, qt.basis(N_photons, 0), ts,
                     c_ops=[np.sqrt(kappa)*a], e_ops=[a])
    print(f"qubit {name}:  master equation <a> = {res.expect[0][-1]:+.5f}   "
          f"one-line formula = {predicted:+.5f}   "
          f"{np.allclose(res.expect[0][-1], predicted, atol=1e-4)}")

# ...and the state really is coherent, so that single number IS the whole cavity.
H = H_ef({'delta_r': delta_r, 'chi': chi, 'epsilon': epsilon, 'sign': -1})
rho = qt.mesolve(H, qt.basis(N_photons, 0), ts, c_ops=[np.sqrt(kappa)*a]).states[-1]
al = qt.expect(a, rho)
print(f"\npurity Tr(rho^2)              = {(rho*rho).tr().real:.6f}")
print(f"fidelity with |alpha = <a>>   = {qt.fidelity(rho, qt.coherent(N_photons, al)):.6f}")

# Break the linearity and it stops being true:
for K in (0.0, 1.0, 5.0):
    rho = qt.mesolve(H + K*adag*adag*a*a, qt.basis(N_photons, 0), ts,
                     c_ops=[np.sqrt(kappa)*a],
                     options={"nsteps": 200000}).states[-1]
    f = qt.fidelity(rho, qt.coherent(N_photons, qt.expect(a, rho)))
    print(f"   with a Kerr term K = {K:4.1f}:  fidelity with a coherent state = {f:.4f}")

## 4. Steady state, and the separation of the two clouds

Set $d\langle a\rangle/dt = 0$:

$$\alpha = \frac{-i\varepsilon}{\kappa/2 + i\Delta_\text{eff}}$$

Probing at the midpoint between the two shifted resonances ($\delta_r = 0$) gives
$\Delta_\text{eff} = \mp\chi$:

$$\alpha_g = \frac{-i\varepsilon}{\kappa/2 - i\chi},
\qquad
\alpha_e = \frac{-i\varepsilon}{\kappa/2 + i\chi}$$

Subtract, over a common denominator $(\kappa/2 + i\chi)(\kappa/2 - i\chi) = \chi^2 + \kappa^2/4$:

$$\alpha_e - \alpha_g
= -i\varepsilon\left[\frac{1}{\kappa/2+i\chi}-\frac{1}{\kappa/2-i\chi}\right]
= -i\varepsilon\,\frac{(\kappa/2 - i\chi)-(\kappa/2+i\chi)}{\chi^2+\kappa^2/4}
= -i\varepsilon\,\frac{-2i\chi}{\chi^2+\kappa^2/4}$$

and since $(-i)(-2i) = 2i^2 = -2$,

$$\boxed{\;\alpha_e - \alpha_g = \frac{-2\varepsilon\,\chi}{\chi^2+\kappa^2/4}\;}$$

The overall minus sign is only a direction in the IQ plane — which cloud sits on which
side. What the fidelity depends on is the modulus,
$|\alpha_e-\alpha_g| = 2\varepsilon\chi/(\chi^2+\kappa^2/4)$.

In [ ]:
# Steady state of  d<a>/dt = -i Delta_eff <a> - i epsilon - (kappa/2) <a>
#   Delta_eff = -delta_r + s*chi,   s = -1 for |g>,  s = +1 for |e>
alpha = lambda s: -1j * epsilon / (kappa/2.0 + 1j * (-delta_r + s*chi))

alpha_g, alpha_e = alpha(-1), alpha(+1)

print("--- Steady-state cavity amplitudes (the cloud centres) ---")
print(f"alpha_g (qubit |g>): {alpha_g.real:+.3f} {alpha_g.imag:+.3f}j   "
      f"|alpha| = {abs(alpha_g):.3f}   phase = {np.angle(alpha_g, deg=True):+7.2f} deg")
print(f"alpha_e (qubit |e>): {alpha_e.real:+.3f} {alpha_e.imag:+.3f}j   "
      f"|alpha| = {abs(alpha_e):.3f}   phase = {np.angle(alpha_e, deg=True):+7.2f} deg")

# Probing at delta_r = 0 the two magnitudes tie by symmetry: the qubit does not change
# how much energy is in the cavity, only the PHASE of the field.
print(f"\nseparation |alpha_e - alpha_g| = {abs(alpha_e-alpha_g):.4f}")
print(f"closed form 2*eps*chi/(chi^2+kappa^2/4) = "
      f"{2*epsilon*chi/(chi**2+kappa**2/4):.4f}")
print(f"maximum attainable 2*eps/kappa         = {2*epsilon/kappa:.4f}  (at chi = kappa/2)")

## 5. Where the separation is largest

### By differentiating

Write $f(\chi) = 2\varepsilon\chi/(\chi^2 + \kappa^2/4)$ and apply the quotient rule
$\left(\frac{u}{v}\right)' = \frac{u'v - uv'}{v^2}$ with $u = 2\varepsilon\chi$ and
$v = \chi^2 + \kappa^2/4$:

$$f'(\chi) = \frac{2\varepsilon\left(\chi^2+\frac{\kappa^2}{4}\right) - 2\varepsilon\chi\,(2\chi)}{\left(\chi^2+\frac{\kappa^2}{4}\right)^2}
= 2\varepsilon\,\frac{\frac{\kappa^2}{4}-\chi^2}{\left(\chi^2+\frac{\kappa^2}{4}\right)^2}$$

> **Watch the sign when expanding.** The factor in front multiplies *both* terms in the
> bracket: $-\left(\chi^2 + \frac{\kappa^2}{4}\right) = -\chi^2 {\,-\,} \frac{\kappa^2}{4}$,
> not $-\chi^2 + \frac{\kappa^2}{4}$. Getting that wrong leaves
> $\chi^2 = -\kappa^2/4$, which has no real solution — a useful symptom that the slip
> happened here.

The denominator never vanishes, so $f' = 0$ requires

$$\frac{\kappa^2}{4} - \chi^2 = 0 \qquad\Longrightarrow\qquad \boxed{\;\chi = \frac{\kappa}{2}\;}$$

It is a maximum, not a minimum: for $\chi < \kappa/2$ the numerator is positive and $f$ is
rising; for $\chi > \kappa/2$ it is negative and $f$ is falling. The value there is

$$f\!\left(\frac{\kappa}{2}\right) = \frac{2\varepsilon\,\frac{\kappa}{2}}{\frac{\kappa^2}{4}+\frac{\kappa^2}{4}} = \frac{2\varepsilon}{\kappa}$$

### Without calculus, and more revealing

Divide numerator and denominator by $\chi$:

$$f(\chi) = \frac{2\varepsilon}{\dfrac{\kappa^2/4}{\chi} + \chi}$$

Maximising $f$ means **minimising** the denominator, a sum of two competing terms: the first
blows up as $\chi \to 0$, the second as $\chi \to \infty$. For positive numbers,
AM $\ge$ GM gives

$$\frac{\kappa^2/4}{\chi} + \chi \;\ge\; 2\sqrt{\frac{\kappa^2}{4}} = \kappa$$

with equality **exactly when the two terms are equal**, $\frac{\kappa^2/4}{\chi} = \chi$,
i.e. $\chi = \kappa/2$. So $f_\text{max} = 2\varepsilon/\kappa$, same answer.

This route names the physics. The $\chi$ upstairs is the **contrast** — how far the
resonances separate. The $\chi^2$ downstairs is the **signal lost** by moving off the peak.
The optimum is where the two contributions balance.

### A geometric way to remember it

At $\chi = \kappa/2$ the two points sit **exactly $90^\circ$ apart** on the resonance circle,
and the separation is the chord of a quarter turn, $\sqrt{2}\,|\alpha|$.

### The caveat that matters in the lab

This optimum holds at **fixed drive** $\varepsilon$. If instead the photon number
$\bar n = |\alpha|^2$ is held fixed — which is the real constraint, since the critical
photon number for transmon ionisation is what limits you — then substituting
$\varepsilon = \sqrt{\bar n\,(\chi^2+\kappa^2/4)}$ gives

$$|\alpha_e-\alpha_g| = \frac{2\chi\sqrt{\bar n}}{\sqrt{\chi^2+\kappa^2/4}}$$

which **increases monotonically** with $\chi$ and saturates at $2\sqrt{\bar n}$: the two
points end up diametrically opposite and the chord becomes the diameter. At fixed photon
number there is no optimum — larger $\chi$ is always better. In practice $\chi \sim \kappa/2$
is still used, for other reasons: staying inside the dispersive limit ($\chi \ll \Delta$),
avoiding Purcell decay, and keeping the response bandwidth.

## The $\chi/\kappa$ trade-off

$\kappa$ is how **wide** the readout resonance is; $\chi$ is how far it **moves** when the
qubit flips. So $\chi/\kappa$ is "how far the resonance shifts, measured in linewidths",
and it is the only dimensionless knob that matters here.

* $\chi \ll \kappa$ — the two Lorentzians overlap. Plenty of signal, no contrast.
* $\chi \gg \kappa$ — probing between them lands in the tail of both. Plenty of contrast,
  no signal.
* $\chi = \kappa/2$ — the peaks sit one linewidth apart and the probe is at half maximum
  of each. Best compromise.



$$|\alpha_e-\alpha_g| = \frac{A\,\chi}{(\kappa/2)^2+\chi^2}
\qquad\Longrightarrow\qquad
\frac{d}{d\chi}=0 \;\Rightarrow\; \chi = \kappa/2$$

The $\chi$ upstairs is the contrast, the $\chi^2$ downstairs is the signal you lose by
moving off the peak.

In [ ]:
def plot_chi_kappa_tradeoff(cases=(0.08, 0.5, 2.5), kappa=1.0, epsilon=0.5, span=4.0):
    """Three regimes of the dispersive readout, plus the trade-off curve.

    cases : the chi/kappa values to show as columns
    kappa : resonator linewidth (sets the units of the frequency axis)
    epsilon : drive amplitude (sets the vertical scale only)
    """
    # intracavity amplitude: a Lorentzian centred at -s*chi, width kappa
    alpha = lambda d, s, chi: -1j*epsilon / (kappa/2 - 1j*(d + s*chi))
    d = np.linspace(-span, span, 2000) * kappa

    fig = plt.figure(figsize=(14, 8.5))
    gs = fig.add_gridspec(3, len(cases), height_ratios=[1, 1, 1.15],
                          hspace=.55, wspace=.3)

    for col, ratio in enumerate(cases):
        chi = ratio * kappa
        ag, ae = alpha(d, -1, chi), alpha(d, +1, chi)      # whole sweep
        g0, e0 = alpha(0, -1, chi), alpha(0, +1, chi)      # at the probe

        # --- row 0: the two resonances, and where we probe -----------------
        ax = fig.add_subplot(gs[0, col])
        ax.plot(d/kappa, np.abs(ag), color="C0", lw=2, label=r"qubit $|g\rangle$")
        ax.plot(d/kappa, np.abs(ae), color="C1", lw=2, label=r"qubit $|e\rangle$")
        ax.axvline(0, color="r", ls="--", lw=1.5)          # the probe tone
        ax.set_title(rf"$\chi/\kappa$ = {ratio:.2f}", fontsize=11, fontweight="bold")
        ax.set_xlabel(r"probe detuning / $\kappa$", fontsize=8)
        ax.set_ylabel(r"$|\alpha|$", fontsize=8)
        ax.grid(alpha=.3)
        if col == 0:
            ax.legend(fontsize=7.5, loc="upper left")

        # --- row 1: the same thing seen in the IQ plane --------------------
        # plotting alpha.real vs alpha.imag traces the resonance CIRCLE,
        # because the sweep variable d is a parameter, not an axis
        ax = fig.add_subplot(gs[1, col])
        ax.plot(ag.real, ag.imag, color="C0", lw=1.5, alpha=.4)
        ax.plot(ae.real, ae.imag, color="C1", lw=1.5, alpha=.4)
        ax.plot(g0.real, g0.imag, "o", color="C0", ms=10)
        ax.plot(e0.real, e0.imag, "o", color="C1", ms=10)
        ax.plot([g0.real, e0.real], [g0.imag, e0.imag], "k-", lw=2.5)
        ax.set_title(f"separation = {abs(e0-g0):.3f}", fontsize=9)
        ax.set_xlabel("I", fontsize=8); ax.set_ylabel("Q", fontsize=8)
        ax.set_aspect("equal")     # circles must look like circles
        ax.grid(alpha=.3)

    # --- row 2: one wide panel with the trade-off curve --------------------
    ax = fig.add_subplot(gs[2, :])                # ':' spans every column
    r = np.linspace(.02, max(cases)*1.2, 600)
    sep = 2*epsilon*(r*kappa) / ((kappa/2)**2 + (r*kappa)**2)
    ax.plot(r, sep, color="C2", lw=2.5)
    for ratio in cases:
        chi = ratio*kappa
        s = 2*epsilon*chi/((kappa/2)**2 + chi**2)
        ax.plot(ratio, s, "o", ms=11, zorder=5)
        ax.annotate(rf"$\chi/\kappa$={ratio}", (ratio, s),
                    textcoords="offset points", xytext=(8, 10), fontsize=9)
    ax.axvline(.5, color="k", ls=":", lw=1.5)
    ax.set_xlabel(r"$\chi/\kappa$  —  how far the resonance moves, in linewidths", fontsize=10)
    ax.set_ylabel(r"$|\alpha_e-\alpha_g|$", fontsize=10)
    ax.set_title(r"Too small: no contrast.   Too large: no signal.   Best at $\chi=\kappa/2$.",
                 fontsize=11)
    ax.grid(alpha=.3)
    return fig


plot_chi_kappa_tradeoff(cases=(0.08, 0.5, 2.5), kappa=kappa, epsilon=epsilon)
plt.show()

## 6. The IQ clouds and the emission matrix

Each cloud is the classical centre $\alpha$ from section 4, smeared by vacuum
fluctuations and amplifier noise. Projecting onto the line joining the two centres
is the matched filter for isotropic Gaussians of equal width; the threshold sits at
the midpoint. Counting which side each shot lands on gives
$F_{g\mathcal{G}}, F_{e\mathcal{E}}$ — the emission matrix $E$ of Eqn. (3) that
`HMM_redout.ipynb` consumes.

The fidelity needs no Monte Carlo: it is $\Phi\!\left(\frac{|\alpha_e-\alpha_g|}{2\sigma}\right)$.
The 5000 shots are for the picture.

In [ ]:
# =====================================================================
# The clouds: classical centre + vacuum noise + amplifier noise
# =====================================================================
n_shots = 5000

sigma_quantum = 0.5   # vacuum fluctuations, variance 1/4 per quadrature
sigma_hemt    = 0.45  # cryogenic HEMT amplifier (the paper's JPA was not used)
sigma_total   = np.sqrt(sigma_quantum**2 + sigma_hemt**2)



# Generate random Gaussian noise centered around the steady-state alphas
np.random.seed(42)
shots_g_I = np.real(alpha_g) + np.random.normal(0, sigma_total, n_shots)
shots_g_Q = np.imag(alpha_g) + np.random.normal(0, sigma_total, n_shots)

shots_e_I = np.real(alpha_e) + np.random.normal(0, sigma_total, n_shots)
shots_e_Q = np.imag(alpha_e) + np.random.normal(0, sigma_total, n_shots)

# =====================================================================
# STEP 4: OPTIMAL PROJECTION & DECISION THRESHOLD
# =====================================================================
# The line of maximum contrast is the vector connecting the two centers:
d_vector = alpha_e - alpha_g
projection_angle = np.angle(d_vector)

# Function to project 2D IQ points onto this optimal 1D axis
def project_to_1d(I, Q, angle):
    # Rotate by the angle of the separation vector to align it to the real axis
    return np.real((I + 1j * Q) * np.exp(-1j * angle))

# Project the clouds and centers
proj_g = project_to_1d(shots_g_I, shots_g_Q, projection_angle)
proj_e = project_to_1d(shots_e_I, shots_e_Q, projection_angle)

proj_center_g = project_to_1d(np.real(alpha_g), np.imag(alpha_g), projection_angle)
proj_center_e = project_to_1d(np.real(alpha_e), np.imag(alpha_e), projection_angle)

# The optimal decision threshold is the midpoint between the two projected centers
threshold = (proj_center_g + proj_center_e) / 2.0

# =====================================================================
# STEP 5: CALCULATING THE HMM EMISSION MATRIX COEFFICIENTS
# =====================================================================
# If proj_center_g is lower than proj_center_e:
# - Correctly read |g> as G if proj < threshold
# - Correctly read |e> as E if proj > threshold
if proj_center_g < proj_center_e:
    correct_g = np.sum(proj_g < threshold)
    correct_e = np.sum(proj_e > threshold)
else:
    correct_g = np.sum(proj_g > threshold)
    correct_e = np.sum(proj_e < threshold)

# Fidelities
F_gG = correct_g / n_shots
F_eE = correct_e / n_shots

# Readout Errors (false classifications)
F_gE = 1.0 - F_gG
F_eG = 1.0 - F_eE

print("--- HMM Emission Matrix Coefficients ---")
print(f"F_gG (Fidelity to identify Ground) : {F_gG:.2%}")
print(f"F_eE (Fidelity to identify Excited): {F_eE:.2%}")
print(f"F_gE (Ground read as Excited)      : {F_gE:.2%}")
print(f"F_eG (Excited read as Ground)      : {F_eG:.2%}\n")

# =====================================================================
# STEP 6: PLOTTING THE CLOUDS AND 1D HISTOGRAMS
# =====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 2D IQ Clouds
ax1.scatter(shots_g_I, shots_g_Q, color='C0', alpha=0.2, s=4, label='Qubit in $|g\\rangle$')
ax1.scatter(shots_e_I, shots_e_Q, color='C1', alpha=0.2, s=4, label='Qubit in $|e\\rangle$')
ax1.plot(alpha_g.real, alpha_g.imag, 'ko', markersize=8, label='Steady-state centers')
ax1.plot(alpha_e.real, alpha_e.imag, 'ko', markersize=8)
ax1.set_title("Simulated 2D IQ Readout Clouds", fontsize=12, fontweight='bold')
ax1.set_xlabel("In-Phase Quadrature I (V)", fontsize=10)
ax1.set_ylabel("Quadrature Q (V)", fontsize=10)
ax1.grid(True, linestyle=':')
ax1.legend(loc='upper right')
ax1.set_aspect('equal')

# Plot 1D Projected Histograms
ax2.hist(proj_g, bins=60, color='C0', alpha=0.6, density=True, label='Qubit in $|g\\rangle$')
ax2.hist(proj_e, bins=60, color='C1', alpha=0.6, density=True, label='Qubit in $|e\\rangle$')
ax2.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Decision Threshold ({threshold:.3f})')
ax2.set_title("1D Projection & HMM Threshold", fontsize=12, fontweight='bold')
ax2.set_xlabel("Projected Quadrature Coordinate (V)", fontsize=10)
ax2.set_ylabel("Probability Density", fontsize=10)
ax2.grid(True, linestyle=':')
ax2.legend()

plt.tight_layout()
plt.show()

## 7. What this model does not include

**No integration time.** The fidelity computed above is that of an instantaneous sample of
the field. A real readout integrates the output for $t_m$, and the separation in units of
the noise grows as $\sqrt{\eta\,\kappa\,t_m}$ — so fidelity is a function of $t_m$, which
never appears here. That trade-off is the central design question for a low-$Q$ storage
cavity, where $t_m$ must be kept short: the false-positive floor of the HMM is
$1 - e^{-t_m/T_1^s}$, so a cavity with a hundred times lower $Q$ demands a readout roughly a
hundred times faster. Dixit's supplement makes the same point for a copper accumulation
cavity, and points at $>99\%$ fidelity in $100\,$ns using a quantum-limited parametric
amplifier and pulse shaping.

**No qubit dynamics during the measurement.** The qubit is frozen in $|g\rangle$ or
$|e\rangle$, so both clouds come out as symmetric Gaussians with equal fidelities. In the
real device a qubit that decays partway through the readout deposits its point *between* the
two blobs, producing a non-Gaussian tail and an asymmetry between $F_{g\mathcal{G}}$ and
$F_{e\mathcal{E}}$.

**Hard thresholding throws away information.** Section 6 projects each shot onto the optimal
axis and then compares it to a threshold. The projected value itself — the *soft
information* — carries roughly twice the information of the resulting bit, because
discarding how far a point lies from the threshold is discarding evidence. Feeding those
projected values to the HMM as a Gaussian likelihood, instead of the $2\times 2$ fidelity
matrix, roughly doubles the slope of the log-odds. See D'Anjou & Coish, PRL 113, 230402
(2014), and the soft-decoding literature in quantum error correction.

**The dispersive Hamiltonian is itself an approximation.** $\chi = g^2/\Delta$ requires
$|\Delta| \gg g$; the full Jaynes-Cummings result falls below it as the qubit approaches the
resonator, by about $7\%$ at $|\Delta| = 20g$ and $25\%$ at $|\Delta| = 10g$.